In [1]:
import pandas as pd
import geopandas as gpd
import sys
import matplotlib.pyplot as plt 
import seaborn as sns

import numpy as np

from shapely import wkt
import seaborn as sns
import infrastructure_utils as iu
dir_path = '/home/qsel/Desktop/data/consumption_data/'

In [2]:

%%time
# Define Projection 
project_crs = 'epsg:32735'
geo_crs = 'epsg:4326'

# EICV RAW DATA
eicv_file = '/home/qsel/Desktop/data/REG_electricity_expenditure/EICV5_Person_Household_Shared_13_02_2023v.csv'
eicv_HH = pd.read_csv(eicv_file, encoding='latin-1', low_memory=False)
# schools
schools = gpd.read_file(dir_path+'Education_Facilities.geojson')
# health centers
health_centers = gpd.read_file(dir_path+'Health_Facilities.geojson')
# trade centres
trade_centres =  gpd.read_file(dir_path+'Trade_centers.geojson')
# coffee centers
coffee_centres =  gpd.read_file(dir_path+'Coffee_Washing_Stations.geojson')
# trade centers
tourism_centres =  gpd.read_file(dir_path+'Tourism_POI.geojson')
# industry parks
industry_centres =  gpd.read_file(dir_path+'Industrial_parks.geojson')
# district offices
district_offices =  gpd.read_file(dir_path+'District_offices.geojson')
# religious centres
religious_centres =  gpd.read_file(dir_path+'Religious_Facilities.geojson')
# sports centers
sports_centres =  gpd.read_file(dir_path+'Sports.geojson')
# major towns
major_towns = pd.read_csv(dir_path+'rwanda_major_towns.csv')
major_towns['geometry'] = major_towns['geometry'].apply(wkt.loads)
major_towns = gpd.GeoDataFrame(major_towns, crs=geo_crs)
# bus stations
bus_stations =  gpd.read_file(dir_path+'Bus_stations.geojson').iloc[:-3,:] # remove rows with weird gps coordinates
# landmark
landmark =  gpd.read_file(dir_path+'national_roads_and_landmarks/Land_Mark.shp')
# district roads
district_road =  gpd.read_file(dir_path+'national_roads_and_landmarks/National_District1_Road.shp')
# National Paved roads
national_paved_road =  gpd.read_file(dir_path+'national_roads_and_landmarks/National_Road_Paved.shp')
# National unPaved roads
national_unpaved_road =  gpd.read_file(dir_path+'national_roads_and_landmarks/National_Road_Unpaved.shp')
# all country road network
road_netwk =  gpd.read_file(dir_path+'Road_network.geojson')
# cell towers
cell_towers = pd.read_csv(dir_path+'rwanda_cell_locations.csv').iloc[:,[0,6,7]]
cell_towers.columns = ["tower_type", "longitude", "latitude"]
cell_towers = gpd.GeoDataFrame(
    cell_towers, geometry=gpd.points_from_xy(cell_towers.longitude, cell_towers.latitude), crs=geo_crs)




## Summarized version (work in progress)
# socio_econ_data = [schools, health_centers, trade_centres, coffee_centres, tourism_centres, industry_centres, district_offices,
#  religious_centres, sports_centres, major_towns, bus_stations, landmark, district_offices, national_paved_road, national_unpaved_road, road_netwk]
# for count,df in enumerate(socio_econ_data):
#     socio_econ_data[count] = df.to_crs(project_crs)

schools = schools.to_crs(project_crs)
health_centers = health_centers.to_crs(project_crs)
trade_centres = trade_centres.to_crs(project_crs)
coffee_centres = coffee_centres.to_crs(project_crs)
tourism_centres = tourism_centres.to_crs(project_crs)
industry_centres = industry_centres.to_crs(project_crs)
district_offices = district_offices.to_crs(project_crs)
religious_centres = religious_centres.to_crs(project_crs)
sports_centres = sports_centres.to_crs(project_crs)
major_towns = major_towns.to_crs(project_crs)
bus_stations = bus_stations.to_crs(project_crs)
landmark = landmark.to_crs(project_crs)
district_road = district_road.to_crs(project_crs)
national_paved_road = national_paved_road.to_crs(project_crs)
national_unpaved_road = national_unpaved_road.to_crs(project_crs)
cell_towers = cell_towers.to_crs(project_crs)
road_netwk = road_netwk.to_crs(project_crs)

# Province towns
kigali = major_towns[major_towns.NomAncien=='Kigali']
southCapital = major_towns[major_towns.NomAncien=='Nyanza']
northCapital = major_towns[major_towns.NomAncien=='Byumba']
westCapital = major_towns[major_towns.NomAncien=='Kibuye']
eastCapital = major_towns[major_towns.NomAncien=='Rwamagana']

survey_gps = iu.get_survey_geopandas_df(eicv_HH, geo_crs, project_crs)
survey_gps.shape

CPU times: user 11.7 s, sys: 415 ms, total: 12.1 s
Wall time: 12.1 s


(14580, 10)

In [3]:
%%time
# add school distances
survey_socio_econ_dists = gpd.sjoin_nearest(survey_gps,
                                                schools[['geometry']], how='left',
                                       distance_col="school_building_dist").drop('index_right',axis=1).rename(columns={'code_vill_':'village_id'}).drop_duplicates(keep='last')
# add health centers distances
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                health_centers[['geometry']], how='left',
                                                 distance_col="health_building_dist").drop('index_right',axis=1).drop_duplicates(keep='last')
# add trade centers distances
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                trade_centres[['geometry']], how='left',
                                                 distance_col="trade_building_dist").drop('index_right',axis=1).drop_duplicates(keep='last')

# add coffee centers distances
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                coffee_centres[['geometry']], how='left',
                                                 distance_col="coffee_building_dist").drop('index_right',axis=1).drop_duplicates(keep='last').drop_duplicates(keep='last')
# add tourism centers distances
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                tourism_centres[['geometry']], how='left',
                                                 distance_col="tourism_building_dist").drop('index_right',axis=1).drop_duplicates(keep='last')

# add industry parks  distances
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                industry_centres[['geometry']], how='left',
                                                 distance_col="industrypark_building_dist").drop('index_right',axis=1).drop_duplicates(keep='last')
# add District_offices  distances
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                industry_centres[['geometry']], how='left',
                                                 distance_col="District_offices_building_dist").drop('index_right',axis=1).drop_duplicates(keep='last')
# add religious_centres distances
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                religious_centres[['geometry']], how='left', 
                                                distance_col="religious_centres_building_dist").drop('index_right',axis=1).drop_duplicates(keep='last')
# add Sports centres distances
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                sports_centres[['geometry']], how='left',
                                                 distance_col="sports_centres_building_dist").drop('index_right',axis=1).drop_duplicates(keep='last')
# add major town distances
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                major_towns[['geometry']], how='left',
                                                 distance_col="major_towns_building_dist").drop('index_right',axis=1).drop_duplicates(keep='last')
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                kigali[['geometry']], how='left',
                                                 distance_col="kigali_building_dist").drop('index_right',axis=1).drop_duplicates(keep='last')
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                southCapital[['geometry']], how='left',
                                                 distance_col="southCapital_building_dist").drop('index_right',axis=1).drop_duplicates(keep='last')
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                northCapital[['geometry']], how='left',
                                                 distance_col="northCapital_building_dist").drop('index_right',axis=1).drop_duplicates(keep='last')
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                westCapital[['geometry']], how='left',
                                                 distance_col="westCapital_building_dist").drop('index_right',axis=1).drop_duplicates(keep='last')
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                eastCapital[['geometry']], how='left',
                                                 distance_col="eastCapital_building_dist").drop('index_right',axis=1).drop_duplicates(keep='last')
# add bus stations distances
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                bus_stations[['geometry']], how='left',
                                                 distance_col="bus_stations_building_dist").drop('index_right',axis=1).drop_duplicates(keep='last')
# add landmark distances
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                landmark[['geometry']], how='left',
                                                 distance_col="landmark_building_dist").drop('index_right',axis=1).drop_duplicates(keep='last')


# add National Paved roads distances
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                national_paved_road[['geometry']], how='left',
                                                 distance_col="national_paved_road_building_dist").drop('index_right',axis=1).drop_duplicates(keep='last')

# add National unPaved roads distances
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                national_unpaved_road[['geometry']], how='left',
                                                 distance_col="national_unpaved_road_building_dist").drop('index_right',axis=1).drop_duplicates(keep='last')

# add cell towers
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                cell_towers[['geometry']], how='left',
                                                 distance_col="cell_tower_building_dist").drop('index_right',axis=1).drop_duplicates(keep='last')

# country wide road network
survey_socio_econ_dists = gpd.sjoin_nearest(survey_socio_econ_dists,
                                                road_netwk[['status','settlement','geometry']], 
                                       how='left', distance_col="road_netwk_building_dist").drop('index_right',axis=1).drop_duplicates(subset=['hhid'],keep='last')

priority_columns = ['hhid','school_building_dist','health_building_dist', 'trade_building_dist', 'coffee_building_dist','tourism_building_dist', 'industrypark_building_dist',
                    'District_offices_building_dist', 'religious_centres_building_dist','sports_centres_building_dist', 'major_towns_building_dist','kigali_building_dist',
                     'southCapital_building_dist','northCapital_building_dist','westCapital_building_dist','eastCapital_building_dist','bus_stations_building_dist',
                    'landmark_building_dist','national_paved_road_building_dist','national_unpaved_road_building_dist', 'cell_tower_building_dist','road_netwk_building_dist']
survey_socio_econ_dists = survey_socio_econ_dists[priority_columns]
distance_cols = survey_socio_econ_dists.drop('hhid',axis=1).columns.tolist()
survey_socio_econ_dists[distance_cols]=survey_socio_econ_dists[distance_cols]/1000
survey_socio_econ_dists.shape

CPU times: user 15.7 s, sys: 7.23 ms, total: 15.7 s
Wall time: 15.7 s


(14580, 22)

### road density

In [11]:
buffer = 200
survey_gps_buffered = survey_gps.copy()
survey_gps_buffered.geometry = survey_gps_buffered.buffer(buffer)

road_density = survey_gps_buffered.sjoin(road_netwk)
total_roads = road_density.groupby(['hhid'])['globalid'].count().reset_index().rename(columns={'globalid':f'total_road_count_in_{buffer}m_radius'})
road_status = road_density.groupby(['hhid','status'])['globalid'].count().reset_index().pivot(index='hhid',columns='status',values='globalid').rename(columns={
    'Paved':'paved_status',
    'Unpaved':'unpaved_status'}).reset_index()
road_class = road_density.groupby(['hhid','class'])['globalid'].count().reset_index().pivot(index='hhid',columns='class',values='globalid').rename(columns={
    'Added Link':'class_added_link',
    'District Road Class 1':'class_district_road_1',
    'Draft District Road Class 2':'class_district_road_2',
    'Footpath':'class_footpath',
    'National Road':'class_national_road',
    'Other road':'class_other_road',
}).reset_index()
classified_road_density = total_roads.merge(road_status,how='left').merge(road_class,how='left')

# merge with distance metrics
socio_econ_metrics =survey_socio_econ_dists.merge(classified_road_density,how='left').fillna(0)
socio_econ_metrics.shape

(14580, 31)

### Road Quality

In [61]:
road_netwk[road_netwk['commentsdi']!=' '].head()

,objectid_1,road_no,settlement,new_id,class,status,district,commentsdi,type,source,capture,globalid,created_user,created_date,last_edited_user,last_edited_date,shape_Length,geometry
806,807,DR 2,Rural,DR 2,District Road Class 1,Unpaved,Rusizi,Section Kibangira-Gikundamvura was rehabilitat...,District road class1,RTDA,Based on Orthophoto from 2009,{17DF16FC-C0C1-4FE9-BEC7-64DC432BF3F5},None,None,None,None,0.000116,"MULTILINESTRING ((726675.226 9699818.210, 7266..."
809,810,DR 2,Rural,DR 2,District Road Class 1,Unpaved,Rusizi,Section Kibangira-Gikundamvura was rehabilitat...,District road class1,RTDA,Based on Orthophoto from 2009,{A545E0CE-E1AE-4096-A659-D6CA3FFC06FC},None,None,None,None,0.000278,"MULTILINESTRING ((726662.349 9699816.876, 7266..."
918,919,DR 2,Rural,DR 2,District Road Class 1,Unpaved,Rusizi,Section Kibangira-Gikundamvura was rehabilitat...,District road class1,RTDA,Based on Orthophoto from 2009,{078C29D4-7EDF-45AC-AEA7-6FE10C6C1EC8},None,None,None,None,0.011351,"MULTILINESTRING ((726631.683 9699820.774, 7266..."
1138,1139,DR 3,Rural,DR 3,District Road Class 1,Paved,Rusizi,"Paved from Bugarama to CIMERWA, the remaining ...",District road class1,RTDA,Based on Orthophoto from 2009,{3E4E05F4-A8B9-4CEF-A356-1CC030EBB85C},None,None,None,None,0.003400,"MULTILINESTRING ((723135.627 9701765.172, 7231..."
1174,1175,DR 3,Rural,DR 3,District Road Class 1,Paved,Rusizi,"Paved from Bugarama to CIMERWA, the remaining ...",District road class1,RTDA,Based on Orthophoto from 2009,{1A624E3B-94A3-47C5-B988-DC5E49EB2DEF},None,None,None,None,0.001004,"MULTILINESTRING ((723173.669 9701869.546, 7231..."


In [45]:
road_netwk.shape

(93623, 18)

In [43]:
road_netwk[road_netwk['class']=='National Road'].iloc[0:2].commentsdi.loc[34]

' '

In [61]:
road_netwk[['status','settlement','geometry']].status.value_counts()

Unpaved    89745
Paved       3878
Name: status, dtype: int64

### Save file

In [13]:

socio_econ_metrics.to_csv('../data/processed/socio_econ_metrics.csv',index=False)